In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random


d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialize tokenizer for token counting
encoding = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer

def count_tokens(text):
    """Count tokens in text using tiktoken"""
    if text is None:
        return 0
    return len(encoding.encode(str(text)))


def convert_to_pandas(dataset, batch_size=1000):
    """
    Convert a Hugging Face dataset (streaming or not) to a pandas DataFrame in batches.
    """
    df_list = []
    
    i=0
    # Hugging Face streaming datasets use iter() for batching
    for batch in dataset.iter(batch_size=batch_size):
        print(f"Processing batch {i} \n")
        i += 1
        # batch is a dict, convert directly to DataFrame
        df_batch = pd.DataFrame(batch)
        df_list.append(df_batch)
    
    df = pd.concat(df_list, ignore_index=True)
    return df

def sample_hf_data(data, sample_size=1000):

    # Shuffle the dataset (buffer_size controls memory usage)
    shuffled = data.shuffle(buffer_size=sample_size)
    # Take random samples
    sampled_dataset = shuffled.take(sample_size)
    
    return sampled_dataset


### Commonsense / Multi-hop Reasoning

tau/commonsense_qa

In [3]:
commonsense_master0=pd.DataFrame()

In [4]:
##This needs to be reasoned with high reasoning
cs1 = load_dataset('tau/commonsense_qa',
                            streaming=True)['train'].select_columns(['question','question_concept','choices','answerKey'])

sample_cs1 = sample_hf_data(cs1, sample_size=2000)
cs1_df = convert_to_pandas(sample_cs1)


cs1_df['answer'] = cs1_df['answerKey']
##Let's format the dataset
cs1_df['input'] = cs1_df['question'] = (
    cs1_df['question'] + 
    "\n\nOptions:\n" + 
    cs1_df['choices'].apply(lambda x: '\n'.join([f"{label}. {text}" for label, text in zip(x['label'], x['text'])]) if isinstance(x, dict) and 'label' in x and 'text' in x else "")
)

cs1_df.rename(columns={'answer':'ground_truth'}, inplace=True)
cs1_df=cs1_df[['input','ground_truth']]

cs1_df['source_answer']=cs1_df['ground_truth']

cs1_df['source'] = 'tau/commonsense_qa'
cs1_df['domain'] ='commonsense_reasoning'
cs1_df['problem_type']='multiple_choice'
cs1_df['question_type']='multiple_choice'
cs1_df['split']='train'
commonsense_master1=pd.concat([commonsense_master0, cs1_df], ignore_index=True)

Processing batch 0 

Processing batch 1 



2.tasksource/strategy-qa

In [5]:
##This is an important dataset for reasoning. Something gemma is bad at

def create_strategyqa_prompt(row):
    facts_text = '\n'.join([f"- {fact}" for fact in row['facts']])
    
    return f"""Question: {row['question']}

Facts:
{facts_text}"""


cs2 = load_dataset('tasksource/strategy-qa',
                            streaming=True)['train'].select_columns(['question','facts','answer'])

sample_cs2 = sample_hf_data(cs2, sample_size=3000)
cs2_df = convert_to_pandas(sample_cs2)



##Let's format the dataset
cs2_df['input'] = cs2_df.apply(create_strategyqa_prompt, axis=1)
cs2_df['ground_truth'] = cs2_df['answer'].astype(str).str.capitalize()

cs2_df=cs2_df[['input','ground_truth']] 
cs2_df['source_answer']=cs2_df['ground_truth']

cs2_df['source'] = 'tasksource/strategy-qa'
cs2_df['domain'] ='commonsense_reasoning'
cs2_df['question_type'] = 'multi-hop'
cs2_df['problem_type']='boolean'
cs2_df['split']='train'
commonsense_master2=pd.concat([commonsense_master1, cs2_df], ignore_index=True)

Processing batch 0 

Processing batch 1 

Processing batch 2 



3.hotpot_qa

In [6]:
##This needs to be reasoned with high reasoning - multi-hop reasoning across documents
cs3 = load_dataset('hotpot_qa', 'fullwiki',
                   streaming=True)['train'].select_columns(['question','answer','type','level','context'])

sample_cs3 = sample_hf_data(cs3, sample_size=3000)
cs3_df = convert_to_pandas(sample_cs3)

# Format context from the dict structure (title + sentences)
def format_hotpot_context(context_dict):
    if not isinstance(context_dict, dict) or 'title' not in context_dict:
        return ""
    contexts = []
    titles = context_dict['title']
    sentences_list = context_dict['sentences']
    for title, sentences in zip(titles, sentences_list):
        context_text = ' '.join(sentences)
        contexts.append(f"{title}: {context_text}")
    return '\n\n'.join(contexts)

cs3_df['input'] = "Context:\n" + cs3_df['context'].apply(format_hotpot_context) + "\n\nQuestion: " + cs3_df['question']

cs3_df.rename(columns={'answer':'ground_truth', 'level':'question_type'}, inplace=True)
cs3_df = cs3_df[['input','ground_truth','question_type']]

cs3_df['source_answer'] = cs3_df['ground_truth']
cs3_df['source'] = 'hotpot_qa'
cs3_df['domain'] = 'commonsense_reasoning'
cs3_df['problem_type'] = 'open_ended'
cs3_df['split'] = 'train'

commonsense_master3 = pd.concat([commonsense_master2, cs3_df], ignore_index=True)

Processing batch 0 

Processing batch 1 

Processing batch 2 



In [7]:
samp=commonsense_master3[commonsense_master3['question_type']=='hard'].sample(1)
print(f"input: {samp['input'].values[0]}")
print(f"ground_truth: {samp['ground_truth'].values[0]}")

input: Context:
Harry Beaumont: Harry Beaumont (February 10, 1888 – December 22, 1966) was an American film director, actor, and screenwriter.  He worked for a variety of production companies including Fox, Goldwyn, Metro, Warner Brothers, and Metro-Goldwyn-Mayer.

Alias a Gentleman: Alias a Gentleman is a 1948 film starring Wallace Beery.  The supporting cast includes Dorothy Patrick, Tom Drake, Gladys George and Sheldon Leonard and the movie was the final one to be directed by Harry Beaumont.

Beatrice Fairfax: Beatrice Fairfax is an American silent film serial directed and produced by Leopold Wharton and Theodore Wharton.  First released on August 7, 1916, the series consists of 15 weekly episodes and features the character of "Beatrice Fairfax" (Grace Darling).  The character was inspired by the popular newspaper advice column "Ask Beatrice Fairfax", which had been the world's first column of its kind when launched in 1898.

How Rastus Gets His Turkey: How Rastus Gets His Turkey is

In [8]:
commonsense_master3.groupby('source').size()

source
hotpot_qa                 3000
tasksource/strategy-qa    2290
tau/commonsense_qa        2000
dtype: int64

In [13]:
commonsense_master3['uid'] = ['commonsense' + str(i) for i in range(1, len(commonsense_master3) + 1)]
commonsense_master3.to_parquet('../data/raw-data/commonsense_base_dataset.parquet')
commonsense_master3.to_csv('../data/raw-data/commonsense_base_dataset.csv', index=False)

### Data Analysis / Quantitative Reasoning
Text-based numerical reasoning, statistics, trends.

In [20]:
dmaster0=pd.DataFrame()

In [21]:
from datasets import load_dataset
import pandas as pd

# Load TAT-QA
tatqa = load_dataset('sarrouche/tat-qa-numeric', split='train', streaming=True)
sample_tatqa = sample_hf_data(tatqa, sample_size=2000)
tatqa_df = convert_to_pandas(sample_tatqa)

def format_tatqa_table(table_str):
    """Format table string into readable format"""
    if not table_str or not isinstance(table_str, str):
        return ""
    
    # Table is already in string format with quoted CSV-like structure
    # Clean it up for readability
    lines = table_str.split('\n')
    formatted = ""
    for line in lines:
        if line.strip():
            formatted += line.strip() + "\n"
    return formatted

def create_tatqa_input(row):
    """Create input with table and question"""
    input_text = format_tatqa_table(row['input'])
    return input_text

# Create formatted columns
tatqa_df['input'] = tatqa_df.apply(create_tatqa_input, axis=1)
tatqa_df['ground_truth'] = tatqa_df['output']  # The equation/formula

# Select final columns
tatqa_df = tatqa_df[['input', 'ground_truth']]
tatqa_df['source_answer'] = tatqa_df['ground_truth']
tatqa_df['source'] = 'ibm/tatqa'
tatqa_df['domain'] = 'numerical_reasoning'
tatqa_df['problem_type'] = 'calculation'
tatqa_df['question_type'] = 'Tabular and Textual Question Answering'
tatqa_df['split'] = 'train'

Processing batch 0 

Processing batch 1 



In [22]:
# Load ConvFinQA
convfinqa = load_dataset('FinGPT/fingpt-convfinqa', split='train', streaming=True)
sample_convfinqa = sample_hf_data(convfinqa, sample_size=10000)
convfinqa_df = convert_to_pandas(sample_convfinqa)

def create_convfinqa_input(row):
    """
    Combine text, table, and question into single input.
    Note: This dataset has conversational history which we may need to handle.
    """
    # The 'input' column already contains formatted text + table + question
    return row['input']

# Create formatted columns
convfinqa_df['input'] = convfinqa_df.apply(create_convfinqa_input, axis=1)
convfinqa_df['ground_truth'] = convfinqa_df['output'].astype(str)

# Select final columns
convfinqa_df = convfinqa_df[['input', 'ground_truth']]
convfinqa_df['source_answer'] = convfinqa_df['ground_truth']
convfinqa_df['source'] = 'FinGPT/fingpt-convfinqa'
convfinqa_df['domain'] = 'financial_reasoning'
convfinqa_df['problem_type'] = 'financial_qa'
convfinqa_df['split'] = 'train'
convfinqa_df['question_type'] = 'Conversational financial QA'
convfinqa_df['input_token_count'] = convfinqa_df['input'].apply(count_tokens)
convfinqa_df = convfinqa_df[convfinqa_df['input_token_count'] < 800]

Processing batch 0 

Processing batch 1 

Processing batch 2 

Processing batch 3 

Processing batch 4 

Processing batch 5 

Processing batch 6 

Processing batch 7 

Processing batch 8 

Processing batch 9 



In [23]:
dmaster0=pd.concat([dmaster0,tatqa_df], ignore_index=True)
dmaster0=pd.concat([dmaster0,convfinqa_df[dmaster0.columns]], ignore_index=True)

In [25]:
dmaster0.sample(5)

,input,ground_truth,source_answer,source,domain,problem_type,question_type,split
143,"Table :\n"""",""2019"",""2018"",""2017""\n""Year ended ...",8.165 - 5.418,8.165 - 5.418,ibm/tatqa,numerical_reasoning,calculation,Tabular and Textual Question Answering,train
912,"Table :\n"""","""",""Year ended December 31,"",""""\n""...",(123.364 - 292.518)/292.518,(123.364 - 292.518)/292.518,ibm/tatqa,numerical_reasoning,calculation,Tabular and Textual Question Answering,train
1811,"Table :\n""increase (decrease)"",""Change between...",3.880-5.742,3.880-5.742,ibm/tatqa,numerical_reasoning,calculation,Tabular and Textual Question Answering,train
2159,performance graph comparison of five-year cumu...,56.82,56.82,FinGPT/fingpt-convfinqa,financial_reasoning,financial_qa,Conversational financial QA,train
3469,table of contents company stock performance th...,254.0,254.0,FinGPT/fingpt-convfinqa,financial_reasoning,financial_qa,Conversational financial QA,train


In [26]:
print(dmaster0.groupby('source').size())
dmaster0['uid'] = ['dquant' + str(i) for i in range(1, len(dmaster0) + 1)]
dmaster0.to_parquet('../data/raw-data/dquant_base_dataset.parquet')
dmaster0.to_csv('../data/raw-data/dquant_base_dataset.csv', index=False)

source
FinGPT/fingpt-convfinqa    1774
ibm/tatqa                  2000
dtype: int64


### Basic Science / Reasoning
Physics, chemistry, biology, earth science, scientific method.

In [33]:
from huggingface_hub import login
from dotenv import load_dotenv
import os
load_dotenv()
login(token=os.getenv('hf_api_key'))

In [38]:
# 1. ARC-Challenge (high school science)
arc = load_dataset('ai2_arc', 'ARC-Challenge', split='train', streaming=True)
sample_arc = sample_hf_data(arc, sample_size=3000)
arc_df = convert_to_pandas(sample_arc)

def create_arc_input(row):
    """Format ARC with question and options"""
    question = row['question']
    choices = row['choices']
    
    options_text = "\n\nOptions:\n"
    labels = choices['label']  # ['A', 'B', 'C', 'D']
    texts = choices['text']
    
    options_text += '\n'.join([f"{label}. {text}" for label, text in zip(labels, texts)])
    
    return f"Question: {question}{options_text}"

arc_df['input'] = arc_df.apply(create_arc_input, axis=1)
arc_df['ground_truth'] = arc_df['answerKey']

arc_df = arc_df[['input', 'ground_truth']]
arc_df['source_answer'] = arc_df['ground_truth']
arc_df['source'] = 'ai2_arc/ARC-Challenge'
arc_df['domain'] = 'science'
arc_df['problem_type'] = 'multiple_choice'
arc_df['question_type'] = 'multiple_choice'
arc_df['split'] = 'train'



# 3. SciQ (general science QA)
sciq = load_dataset('sciq', split='train', streaming=True)
sample_sciq = sample_hf_data(sciq, sample_size=3000)
sciq_df = convert_to_pandas(sample_sciq)

def create_sciq_input(row):
    """Format SciQ with question and 4 choices"""
    # SciQ has: correct_answer, distractor1, distractor2, distractor3
    # Need to randomize order and track correct label
    
    question = row['question']
    
    # Create options (SciQ gives them separately)
    options = [
        row['correct_answer'],
        row['distractor1'], 
        row['distractor2'],
        row['distractor3']
    ]
    
    # For simplicity, put correct answer first (will be A)
    # In production, you'd want to shuffle
    options_text = "\n\nOptions:\n"
    for i, opt in enumerate(options):
        label = chr(65 + i)  # A, B, C, D
        options_text += f"{label}. {opt}\n"
    
    return f"Question: {question}{options_text}"

sciq_df['input'] = sciq_df.apply(create_sciq_input, axis=1)
sciq_df['ground_truth'] = 'A'  # Since we put correct answer first

sciq_df = sciq_df[['input', 'ground_truth']]
sciq_df['source_answer'] = sciq_df['ground_truth']
sciq_df['source'] = 'sciq'
sciq_df['domain'] = 'science'
sciq_df['problem_type'] = 'multiple_choice'
sciq_df['question_type'] = 'multiple_choice'
sciq_df['split'] = 'train'


def convert_to_pandas_flexible(dataset, batch_size=1000):
    """
    Convert HF dataset to pandas with schema flexibility for inconsistent batches.
    """
    df_list = []
    i = 0
    
    for batch in dataset.iter(batch_size=batch_size):
        print(f"Processing batch {i}")
        i += 1
        
        # Convert batch dict to DataFrame
        df_batch = pd.DataFrame(batch)
        df_list.append(df_batch)
    
    # Concatenate with flexible schema handling
    if len(df_list) > 0:
        df = pd.concat(df_list, ignore_index=True, sort=False)
        return df
    return pd.DataFrame()

# Use it for GPQA
gpqa = load_dataset('Idavidrein/gpqa', 'gpqa_main', split='train', streaming=True)
sample_gpqa = sample_hf_data(gpqa, sample_size=800)
gpqa_df = pd.DataFrame(sample_gpqa)

# Now process GPQA
def create_gpqa_input(row):
    """Format GPQA - answers are NOT pre-shuffled, need to randomize"""
    question = row['Question']
    correct = row['Correct Answer']
    incorrect = [
        row['Incorrect Answer 1'],
        row['Incorrect Answer 2'],
        row['Incorrect Answer 3']
    ]
    
    # Combine and shuffle
    import random
    all_answers = [correct] + incorrect
    # For reproducibility in training, use a simple shuffle based on question hash
    # In production you'd want proper randomization
    random.seed(hash(question) % 1000)
    random.shuffle(all_answers)
    
    # Find correct answer position
    correct_idx = all_answers.index(correct)
    correct_label = chr(65 + correct_idx)  # A, B, C, or D
    
    options_text = "\n\nOptions:\n"
    for i, ans in enumerate(all_answers):
        label = chr(65 + i)
        options_text += f"{label}. {ans}\n"
    
    return f"Question: {question}{options_text}", correct_label

# Apply formatting
gpqa_df[['input', 'ground_truth']] = gpqa_df.apply(
    lambda row: pd.Series(create_gpqa_input(row)), 
    axis=1
)

# Final columns
gpqa_df = gpqa_df[['input', 'ground_truth']]
gpqa_df['source_answer'] = gpqa_df['ground_truth']
gpqa_df['source'] = 'Idavidrein/gpqa'
gpqa_df['domain'] = 'science'
gpqa_df['problem_type'] = 'multiple_choice'
gpqa_df['question_type'] = 'multiple_choice'
gpqa_df['split'] = 'train'

# Combine all science datasets
science_master = pd.concat([arc_df, sciq_df, gpqa_df], ignore_index=True)
print(f"Total science samples: {len(science_master)}")

Processing batch 0 

Processing batch 1 

Processing batch 0 

Processing batch 1 

Processing batch 2 

Total science samples: 4567


In [41]:

# Check GPQA token counts
science_master['input_token_count'] = science_master['input'].apply(count_tokens)
print(science_master[science_master['source'] == 'Idavidrein/gpqa']['input_token_count'].describe())

# Filter if needed
science_master = science_master[science_master['input_token_count'] < 1000].copy()
print(science_master.groupby('source').size()) 

count     448.000000
mean      203.912946
std       167.328326
min        40.000000
25%       123.000000
50%       173.000000
75%       234.250000
max      2768.000000
Name: input_token_count, dtype: float64
source
Idavidrein/gpqa           447
ai2_arc/ARC-Challenge    1119
sciq                     3000
dtype: int64


In [42]:
print(science_master.groupby('source').size())
science_master['uid'] = ['science' + str(i) for i in range(1, len(science_master) + 1)]
science_master.to_parquet('../data/raw-data/science_base_dataset.parquet')
science_master.to_csv('../data/raw-data/science_base_dataset.csv', index=False)

source
Idavidrein/gpqa           447
ai2_arc/ARC-Challenge    1119
sciq                     3000
dtype: int64


### Creative Writing / Storytelling
Narrative generation, character development, plot, style.

In [66]:
writing = load_dataset('euclaise/writingprompts', split='train', streaming=True)
sample_writing = sample_hf_data(writing, sample_size=2700)
writing_df = convert_to_pandas_flexible(sample_writing)

def create_writing_input(row):
    """Format writing prompt"""
    prompt = row['prompt']
    # Clean the prompt (often has [WP] tags)
    prompt = prompt.replace('[ WP ]', '').replace('[ EU ]', '').replace('[ IP ]', '').strip()
    
    return f"Write a creative story based on this prompt:\n\n{prompt}"

writing_df['input'] = writing_df.apply(create_writing_input, axis=1)
writing_df['ground_truth'] = writing_df['story']  # Human-written story


writing_df = writing_df[['input', 'ground_truth']]
writing_df['source_answer'] = writing_df['ground_truth']
writing_df['source'] = 'euclaise/writingprompts'
writing_df['domain'] = 'creative_writing'
writing_df['problem_type'] = 'generation'
writing_df['question_type'] = 'story_writing'
writing_df['split'] = 'train'

Processing batch 0
Processing batch 1
Processing batch 2


In [67]:
rocstories = load_dataset('igormorgado/ROCStories2018', split='train', streaming=True)
sample_roc = sample_hf_data(rocstories, sample_size=2000)
roc_df = convert_to_pandas_flexible(sample_roc)

def create_roc_input(row):
    """Format ROC story - give first 4 sentences, ask for 5th"""
    context = f"{row['sentence1']} {row['sentence2']} {row['sentence3']} {row['sentence4']}"
    
    return f"Continue this story with a concluding sentence:\n\n{context}"

roc_df['input'] = roc_df.apply(create_roc_input, axis=1)
roc_df['ground_truth'] = roc_df['sentence5']

roc_df = roc_df[['input', 'ground_truth']]
roc_df['source_answer'] = roc_df['ground_truth']
roc_df['source'] = 'igormorgado/ROCStories2018'
roc_df['domain'] = 'creative_writing'
roc_df['problem_type'] = 'story_completion'
roc_df['question_type'] = 'story_completion'
roc_df['split'] = 'train'

writing_df=pd.concat([writing_df, roc_df], ignore_index=True)

# Filter by length (stories can be VERY long)
writing_df['story_token_count'] = writing_df['ground_truth'].apply(count_tokens)
writing_df = writing_df[writing_df['story_token_count'] < 800].copy()




Processing batch 0
Processing batch 1


In [68]:
print(writing_df.groupby('source').size()) 
writing_df.sample(5)

source
euclaise/writingprompts       1846
igormorgado/ROCStories2018    2000
dtype: int64


,input,ground_truth,source_answer,source,domain,problem_type,question_type,split,story_token_count
4189,Continue this story with a concluding sentence...,Jean had a great time with her friends that ni...,Jean had a great time with her friends that ni...,igormorgado/ROCStories2018,creative_writing,story_completion,story_completion,train,11
3922,Continue this story with a concluding sentence...,But Brenda continued to blog because she found...,But Brenda continued to blog because she found...,igormorgado/ROCStories2018,creative_writing,story_completion,story_completion,train,13
4677,Continue this story with a concluding sentence...,Chris called Triple A to fix his car.,Chris called Triple A to fix his car.,igormorgado/ROCStories2018,creative_writing,story_completion,story_completion,train,9
3827,Continue this story with a concluding sentence...,Everything seemed automatically better.,Everything seemed automatically better.,igormorgado/ROCStories2018,creative_writing,story_completion,story_completion,train,5
813,Write a creative story based on this prompt:\n...,It'd been years since the last aircraft flew. ...,It'd been years since the last aircraft flew. ...,euclaise/writingprompts,creative_writing,generation,story_writing,train,606


In [61]:
print(writing_df[['input','ground_truth']].sample(1).values[0])

["Write a creative story based on this prompt:\n\nAn alien species is running out of some kind of resource , luckily they heard you can get anything from an Earth item called an `` Everything Bagel '' ."
 'In all honesty, the invasion was a bit of a letdown. Sure, the weeks leading up to it were interesting. Ever since NASA had announced that an object moving at an unthinkable speed was heading straight towards Earth, it appeared as if the entire world had been knocked off its rocker. Speculation ran wild. Was it aliens? Probably, an unassuming UN spokesperson had announced to the world one Tuesday morning. Conspiracy theorists everywhere rejoiced. Google searches or the benefits of tin foil hats skyrocketed. You could hardly walk through the park without someone somebody asking if you had a moment to talk about our Lord and Savior, Aten the Sun Disk. While the UN prepared speeches and practiced their handshakes, militaries around the globe united to point every sharp, explosive, or va

In [52]:
writing_df.count()

input            1368
ground_truth     1368
source_answer    1368
source           1368
domain           1368
problem_type     1368
split            1368
dtype: int64

In [69]:
writing_df['uid'] = ['writing' + str(i) for i in range(1, len(writing_df) + 1)]
writing_df.to_parquet('../data/raw-data/writing_base_dataset.parquet')
writing_df.to_csv('../data/raw-data/writing_base_dataset.csv', index=False)


### Creative ideation

In [74]:
import random
import pandas as pd

# Expanded and refined ideation prompt templates
ideation_templates = {
    'problem_solving': [
        "Brainstorm 5 innovative solutions to {problem}. Focus on feasibility and impact.",
        "What are unconventional approaches to solve {problem}? Think beyond traditional methods.",
        "Generate creative solutions for {problem} that could be implemented within a year.",
        "Propose both high-tech and low-tech solutions to {problem}.",
        "How might different industries approach solving {problem}? Compare 3 perspectives."
    ],
    'what_if': [
        "What if {scenario}? Explore 4-5 societal, economic, and technological implications.",
        "Imagine a world where {scenario}. What would change in daily life, work, and relationships?",
        "Consider this scenario: {scenario}. What opportunities and challenges would emerge?",
        "If {scenario} became reality tomorrow, what would be the first domino effects?",
        "Explore the long-term consequences: What if {scenario}? Consider 5-year and 20-year impacts."
    ],
    'product_innovation': [
        "Design an innovative {product_type} for {target_audience} that addresses {problem}.",
        "Brainstorm features for a revolutionary {product_type} focused on {use_case}. What makes it 10x better?",
        "Create a product concept that combines {feature1} and {feature2} to help with {use_case}.",
        "What would a {product_type} look like if designed specifically for {target_audience} in {context}?",
        "Reimagine {existing_product} for the year 2030. What features would it need?"
    ],
    'creative_uses': [
        "List 8-10 unexpected uses for {object} in {context}.",
        "How could {object} be creatively repurposed to address {problem}?",
        "Brainstorm innovative applications of {object} that don't exist yet.",
        "What if {object} was the only tool available to solve {problem}? Generate creative solutions.",
        "Combine {object1} with {object2} to create something entirely new. What could it be?"
    ],
    'business_innovation': [
        "Generate 3 disruptive business ideas that combine {concept1} with {concept2}.",
        "What startup ideas could transform {industry} using {technology}?",
        "Brainstorm innovative business models for {industry} that focus on {value_prop}.",
        "How could {industry} adopt practices from {other_industry} to innovate?",
        "Design a subscription service that brings {concept} to {target_market}."
    ],
    'future_scenarios': [
        "Predict 5 ways {technology} will change {domain} by 2035.",
        "What innovations in {domain} are most likely in the next decade? Explain your reasoning.",
        "How might {trend} reshape {industry}? Consider both positive and negative outcomes.",
        "Envision {domain} in 2040. What technologies, practices, and challenges will dominate?",
        "If {technology} advances 10x faster than expected, how would {industry} transform?"
    ],
    'improvement_ideas': [
        "How could we make {activity} 3x more effective or enjoyable?",
        "Brainstorm 5 ways to improve {system} for {users}. Prioritize accessibility.",
        "What small changes to {process} could yield dramatic improvements?",
        "Redesign {experience} from scratch. What would you keep, remove, or add?",
        "How might AI enhance {activity} without replacing the human element?"
    ],
    'system_design': [
        "Design a better system for {process} that addresses current pain points.",
        "How would you rebuild {system} to be more {quality}?",
        "Create a framework for {activity} that scales from individual to organizational level.",
        "What would an ideal {system} look like if designed with {principle} as the priority?",
        "Propose a new model for {domain} that balances {constraint1} and {constraint2}."
    ],
    'creative_combination': [
        "What happens when you combine {concept1}, {concept2}, and {concept3}? Explore possibilities.",
        "Create something new by merging ideas from {domain1} and {domain2}.",
        "Design a hybrid solution that takes the best of {approach1} and {approach2}.",
        "What innovative services emerge when {industry1} collaborates with {industry2}?",
        "Blend {style1} with {style2} to create a unique approach to {activity}."
    ],
    'unconventional_thinking': [
        "What's the opposite of conventional wisdom about {topic}? Explore that angle.",
        "Challenge assumptions: What if the problem of {issue} is actually an opportunity?",
        "Apply {method} from {field} to solve challenges in {different_field}.",
        "What would a child's approach to {problem} look like? Think with fresh perspective.",
        "Reverse engineer success: If {goal} was already achieved, what steps led there?"
    ]
}

# Expanded content pools
problems = [
    "reducing food waste in restaurants", "making online learning more engaging",
    "helping elderly people stay connected", "reducing plastic packaging",
    "improving urban air quality", "making exercise more fun for kids",
    "reducing water consumption in agriculture", "combating misinformation online",
    "improving mental health access", "reducing commute time in cities",
    "making healthy food more affordable", "increasing voter participation",
    "reducing energy consumption in homes", "improving workplace diversity",
    "making public transit more reliable", "reducing student loan burden"
]

scenarios = [
    "humans could photosynthesize like plants", "sleep was optional for humans",
    "everyone could only speak one language", "internet connectivity became free worldwide",
    "artificial gravity was affordable", "all vehicles became autonomous overnight",
    "teleportation was possible within 100 miles", "humans could live underwater",
    "personal AI assistants were mandated by law", "work weeks became 4 days globally",
    "cash money was completely eliminated", "education became fully personalized by AI",
    "aging could be slowed by 50%", "memory could be digitally backed up",
    "energy became 100% renewable", "privacy became impossible to maintain"
]

objects = [
    "shipping containers", "coffee grounds", "old smartphones", "bicycle wheels",
    "wine corks", "cardboard boxes", "laser pointers", "QR codes", "mirrors",
    "magnets", "rope", "solar panels", "batteries", "sensors", "parachutes"
]

technologies = [
    "AI and machine learning", "blockchain", "VR/AR", "quantum computing",
    "gene editing", "robotics", "3D printing", "drones", "IoT sensors",
    "renewable energy", "nanotechnology", "brain-computer interfaces"
]

industries = [
    "healthcare", "education", "transportation", "agriculture", "retail",
    "entertainment", "real estate", "finance", "food service", "manufacturing",
    "construction", "hospitality", "legal services", "energy", "telecommunications"
]

target_audiences = [
    "busy professionals", "college students", "parents with young children",
    "elderly people", "remote workers", "small business owners", "teachers",
    "healthcare workers", "urban residents", "rural communities", "teenagers"
]

activities = [
    "studying", "exercising", "commuting", "grocery shopping", "cooking",
    "networking", "job searching", "learning new skills", "reading",
    "meditating", "team collaboration", "public speaking", "time management"
]

systems = [
    "public transit", "healthcare delivery", "education system", "voting process",
    "waste management", "food distribution", "emergency response", "hiring process",
    "customer service", "supply chain", "urban planning", "justice system"
]

qualities = [
    "equitable", "sustainable", "accessible", "efficient", "transparent",
    "user-friendly", "resilient", "inclusive", "affordable", "scalable"
]

contexts = [
    "emergency situations", "developing countries", "space exploration",
    "underwater environments", "educational settings", "healthcare",
    "entertainment", "disaster relief", "sustainable living", "remote areas"
]

def generate_ideation_prompts(n_samples=2000, seed=42):
    """Generate diverse, high-quality creative ideation prompts"""
    random.seed(seed)
    prompts = []
    
    # Calculate samples per category for balanced distribution
    categories = list(ideation_templates.keys())
    samples_per_category = n_samples // len(categories)
    
    for category in categories:
        for _ in range(samples_per_category):
            template = random.choice(ideation_templates[category])
            
            # Prepare all possible replacements
            replacements = {
                'problem': random.choice(problems),
                'scenario': random.choice(scenarios),
                'product_type': random.choice(['app', 'device', 'service', 'platform', 'tool']),
                'target_audience': random.choice(target_audiences),
                'use_case': random.choice(['productivity', 'health', 'learning', 'connection', 'entertainment']),
                'feature1': random.choice(['AI-powered', 'voice-controlled', 'gamified', 'personalized']),
                'feature2': random.choice(['social', 'data-driven', 'eco-friendly', 'minimal']),
                'object': random.choice(objects),
                'object1': random.choice(objects[:8]),
                'object2': random.choice(objects[7:]),
                'technology': random.choice(technologies),
                'industry': random.choice(industries),
                'industry1': random.choice(industries[:8]),
                'industry2': random.choice(industries[7:]),
                'other_industry': random.choice(['hospitality', 'gaming', 'sports', 'fashion']),
                'concept1': random.choice(['AI', 'sharing economy', 'gamification', 'personalization', 'automation']),
                'concept2': random.choice(['sustainability', 'wellness', 'education', 'community', 'creativity']),
                'concept3': random.choice(['AI', 'social media', 'blockchain', 'VR']),
                'concept': random.choice(['mindfulness', 'lifelong learning', 'circular economy', 'remote collaboration']),
                'value_prop': random.choice(['sustainability', 'accessibility', 'personalization', 'community']),
                'domain': random.choice(['healthcare', 'education', 'work', 'entertainment', 'government']),
                'domain1': random.choice(['technology', 'psychology', 'design', 'economics']),
                'domain2': random.choice(['education', 'healthcare', 'urban planning', 'entertainment']),
                'trend': random.choice(['remote work', 'climate action', 'aging population', 'urbanization', 'AI adoption']),
                'activity': random.choice(activities),
                'system': random.choice(systems),
                'process': random.choice(['onboarding', 'decision-making', 'communication', 'evaluation']),
                'users': random.choice(['beginners', 'experts', 'elderly users', 'children', 'non-technical users']),
                'experience': random.choice(['airport security', 'doctor visit', 'online shopping', 'job interview']),
                'quality': random.choice(qualities),
                'context': random.choice(contexts),
                'existing_product': random.choice(['smartphone', 'calendar', 'email', 'notebook', 'alarm clock']),
                'field': random.choice(['biology', 'architecture', 'music', 'sports']),
                'different_field': random.choice(['business', 'education', 'healthcare', 'government']),
                'approach1': random.choice(['top-down', 'data-driven', 'user-centric', 'agile']),
                'approach2': random.choice(['bottom-up', 'intuitive', 'systematic', 'iterative']),
                'style1': random.choice(['minimalist', 'playful', 'professional', 'artistic']),
                'style2': random.choice(['technical', 'conversational', 'visual', 'interactive']),
                'topic': random.choice(['productivity', 'leadership', 'innovation', 'wellness']),
                'issue': random.choice(['complexity', 'cost', 'time constraints', 'resistance to change']),
                'goal': random.choice(['zero waste', '100% renewable energy', 'universal education', 'carbon neutrality']),
                'method': random.choice(['design thinking', 'agile methodology', 'lean startup', 'systems thinking']),
                'principle': random.choice(['user privacy', 'environmental impact', 'social equity', 'transparency']),
                'constraint1': random.choice(['cost', 'speed', 'quality', 'scalability']),
                'constraint2': random.choice(['user experience', 'security', 'sustainability', 'accessibility']),
                'target_market': random.choice(['busy professionals', 'families', 'students', 'small businesses']),
                'social_issue': random.choice(['social isolation', 'digital literacy', 'climate awareness']),
                'challenge': random.choice(['affordable housing', 'food security', 'mental health support'])
            }
            
            # Format template with available replacements
            try:
                prompt = template.format(**replacements)
            except KeyError as e:
                # Fallback if missing placeholder
                print(f"Warning: Missing placeholder {e} in template: {template}")
                continue
            
            prompts.append({
                'input': prompt,
                'category': category,
                'domain': 'creative_ideation'
            })
    
    # Fill remaining samples to reach n_samples exactly
    remaining = n_samples - len(prompts)
    for _ in range(remaining):
        category = random.choice(categories)
        template = random.choice(ideation_templates[category])
        
        # Generate new replacements for remaining samples
        replacements = {
            'problem': random.choice(problems),
            'scenario': random.choice(scenarios),
            'product_type': random.choice(['app', 'device', 'service', 'platform', 'tool']),
            'target_audience': random.choice(target_audiences),
            'use_case': random.choice(['productivity', 'health', 'learning', 'connection', 'entertainment']),
            'feature1': random.choice(['AI-powered', 'voice-controlled', 'gamified', 'personalized']),
            'feature2': random.choice(['social', 'data-driven', 'eco-friendly', 'minimal']),
            'object': random.choice(objects),
            'object1': random.choice(objects[:8]),
            'object2': random.choice(objects[7:]),
            'technology': random.choice(technologies),
            'industry': random.choice(industries),
            'industry1': random.choice(industries[:8]),
            'industry2': random.choice(industries[7:]),
            'other_industry': random.choice(['hospitality', 'gaming', 'sports', 'fashion']),
            'concept1': random.choice(['AI', 'sharing economy', 'gamification', 'personalization', 'automation']),
            'concept2': random.choice(['sustainability', 'wellness', 'education', 'community', 'creativity']),
            'concept3': random.choice(['AI', 'social media', 'blockchain', 'VR']),
            'concept': random.choice(['mindfulness', 'lifelong learning', 'circular economy', 'remote collaboration']),
            'value_prop': random.choice(['sustainability', 'accessibility', 'personalization', 'community']),
            'domain': random.choice(['healthcare', 'education', 'work', 'entertainment', 'government']),
            'domain1': random.choice(['technology', 'psychology', 'design', 'economics']),
            'domain2': random.choice(['education', 'healthcare', 'urban planning', 'entertainment']),
            'trend': random.choice(['remote work', 'climate action', 'aging population', 'urbanization', 'AI adoption']),
            'activity': random.choice(activities),
            'system': random.choice(systems),
            'process': random.choice(['onboarding', 'decision-making', 'communication', 'evaluation']),
            'users': random.choice(['beginners', 'experts', 'elderly users', 'children', 'non-technical users']),
            'experience': random.choice(['airport security', 'doctor visit', 'online shopping', 'job interview']),
            'quality': random.choice(qualities),
            'context': random.choice(contexts),
            'existing_product': random.choice(['smartphone', 'calendar', 'email', 'notebook', 'alarm clock']),
            'field': random.choice(['biology', 'architecture', 'music', 'sports']),
            'different_field': random.choice(['business', 'education', 'healthcare', 'government']),
            'approach1': random.choice(['top-down', 'data-driven', 'user-centric', 'agile']),
            'approach2': random.choice(['bottom-up', 'intuitive', 'systematic', 'iterative']),
            'style1': random.choice(['minimalist', 'playful', 'professional', 'artistic']),
            'style2': random.choice(['technical', 'conversational', 'visual', 'interactive']),
            'topic': random.choice(['productivity', 'leadership', 'innovation', 'wellness']),
            'issue': random.choice(['complexity', 'cost', 'time constraints', 'resistance to change']),
            'goal': random.choice(['zero waste', '100% renewable energy', 'universal education', 'carbon neutrality']),
            'method': random.choice(['design thinking', 'agile methodology', 'lean startup', 'systems thinking']),
            'principle': random.choice(['user privacy', 'environmental impact', 'social equity', 'transparency']),
            'constraint1': random.choice(['cost', 'speed', 'quality', 'scalability']),
            'constraint2': random.choice(['user experience', 'security', 'sustainability', 'accessibility']),
            'target_market': random.choice(['busy professionals', 'families', 'students', 'small businesses']),
            'social_issue': random.choice(['social isolation', 'digital literacy', 'climate awareness']),
            'challenge': random.choice(['affordable housing', 'food security', 'mental health support'])
        }
        
        try:
            prompt = template.format(**replacements)
            prompts.append({
                'input': prompt,
                'category': category,
                'domain': 'creative_ideation'
            })
        except:
            pass
    
    return pd.DataFrame(prompts)

# Generate dataset
ideation_df = generate_ideation_prompts(n_samples=2000)

ideation_df['ground_truth'] = None  # OSS-120B will generate
ideation_df['source_answer'] = None
ideation_df['source'] = 'synthetic_ideation_prompts'
ideation_df['problem_type'] = 'brainstorming'
ideation_df['question_type'] = 'open_ended'
ideation_df['split'] = 'train'

print(f"✓ Creative ideation prompts generated: {len(ideation_df)}")
print(f"\n✓ Category distribution:")
print(ideation_df['category'].value_counts())
print("\n📝 Sample prompts from each category:")
for cat in ideation_df['category'].unique():
    sample = ideation_df[ideation_df['category'] == cat]['input'].iloc[0]
    print(f"\n{cat.upper()}:")
    print(f"  {sample}")

✓ Creative ideation prompts generated: 2000

✓ Category distribution:
category
problem_solving            200
what_if                    200
product_innovation         200
creative_uses              200
business_innovation        200
future_scenarios           200
improvement_ideas          200
system_design              200
creative_combination       200
unconventional_thinking    200
Name: count, dtype: int64

📝 Sample prompts from each category:

PROBLEM_SOLVING:
  Brainstorm 5 innovative solutions to reducing food waste in restaurants. Focus on feasibility and impact.

WHAT_IF:
  Consider this scenario: privacy became impossible to maintain. What opportunities and challenges would emerge?

PRODUCT_INNOVATION:
  Design an innovative tool for teenagers that addresses increasing voter participation.

CREATIVE_USES:
  Brainstorm innovative applications of mirrors that don't exist yet.

BUSINESS_INNOVATION:
  Design a subscription service that brings lifelong learning to students.

FUTU

In [75]:
ideation_df.sample(5)

,input,category,domain,ground_truth,source_answer,source,problem_type,question_type,split
1754,Blend artistic with visual to create a unique ...,creative_combination,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train
1535,What would an ideal urban planning look like i...,system_design,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train
655,How could QR codes be creatively repurposed to...,creative_uses,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train
1067,Predict 5 ways AI and machine learning will ch...,future_scenarios,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train
253,Consider this scenario: personal AI assistants...,what_if,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train


In [76]:
ideation_df['uid'] = ['ideation' + str(i) for i in range(1, len(ideation_df) + 1)]
ideation_df.to_parquet('../data/raw-data/ideation_base_dataset.parquet')
ideation_df.to_csv('../data/raw-data/ideation_base_dataset.csv', index=False)

### Summarization
Compression, key extraction, abstractive/extractive, various lengths.

In [81]:
# 1. CNN/DailyMail - News summarization
cnn_dm = load_dataset('abisee/cnn_dailymail', '3.0.0', split='train', streaming=True)
sample_cnn = sample_hf_data(cnn_dm, sample_size=3000)
cnn_df = convert_to_pandas_flexible(sample_cnn)

def create_cnn_input(row):
    """Format CNN/DM for summarization"""
    article = row['article']
    return f"Summarize the following news article:\n\n{article}"

cnn_df['input'] = cnn_df.apply(create_cnn_input, axis=1)
cnn_df['ground_truth'] = cnn_df['highlights']

# Filter by article length
cnn_df['article_token_count'] = cnn_df['article'].apply(count_tokens)
cnn_df = cnn_df[cnn_df['article_token_count'] < 1000].copy()

cnn_df = cnn_df[['input', 'ground_truth']]
cnn_df['source_answer'] = cnn_df['ground_truth']
cnn_df['source'] = 'abisee/cnn_dailymail'
cnn_df['domain'] = 'summarization'
cnn_df['problem_type'] = 'news_summary'
cnn_df['split'] = 'train'
cnn_df['question_type'] = 'summarization'

# Combine all summarization datasets
summarization_master = cnn_df

print(f"✓ Total summarization samples: {len(summarization_master)}")
print(f"\n✓ Distribution by source:")
print(summarization_master['source'].value_counts())


Processing batch 0
Processing batch 1
Processing batch 2
✓ Total summarization samples: 2180

✓ Distribution by source:
source
abisee/cnn_dailymail    2180
Name: count, dtype: int64


In [83]:
print(summarization_master[['input']].sample(1).values[0])

['Summarize the following news article:\n\nNEW YORK (CNNMoney.com) -- Is cheese the answer for Cayuga County, New York? Larry Rosenbaum surveys the field where he hopes to build a high-end specialty cheese factory. Like small towns all across America, this agricultural community is suffering, with unemployment approaching 10 percent. Entrepreneur Larry Rosenbaum thinks he can do his part to turn things around. For a decade, the insurance man by trade has been dreaming of building a factory for high-end specialty cheese. One key selling point: His product would meet the strictest standards of the Jewish and Muslim faiths. Rosenbaum says the demand for kosher and halal cheese is high but the selection is slim. So he\'s been eyeing a plot of barren farmland between Aurelius and Auburn -- two Cayuga towns -- as the future home of a $40 million, 64,000-square-foot factory that would churn out feta and brie. The goal is for his company, Saratoga Cheese Corp., to produce 30 million pounds of 

In [84]:
summarization_master['uid'] = ['summarization' + str(i) for i in range(1, len(summarization_master) + 1)]
summarization_master.to_parquet('../data/raw-data/summarization_base_dataset.parquet')
summarization_master.to_csv('../data/raw-data/summarization_base_dataset.csv', index=False)

### Knowledge

- MultiTask Knowledge (MMLU)

In [7]:
from datasets import load_dataset
import json
from typing import List, Dict
import random
import argparse

random.seed(42)

def sample_hf_streaming(dataset_name: str, config: str, split: str, n_samples: int, buffer_size: int = 10000):
    """Sample from HuggingFace dataset using streaming"""
    print(f"    Streaming {n_samples} samples...", end=" ", flush=True)
    dataset = load_dataset(dataset_name, config, split=split, streaming=True)
    shuffled = dataset.shuffle(buffer_size=buffer_size, seed=42)
    
    samples = []
    for i, item in enumerate(shuffled):
        samples.append(item)
        if (i + 1) % 50 == 0:
            print(f"{i+1}", end="...", flush=True)
        if len(samples) >= n_samples:
            break
    
    print(f" Done!")
    return samples

In [16]:
from datasets import load_dataset, concatenate_datasets

subjects = ['anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 
            'college_biology', 'college_chemistry', 'college_computer_science', 
            'college_mathematics', 'college_medicine', 'college_physics',
            'computer_security', 'conceptual_physics', 'econometrics', 
            'electrical_engineering', 'formal_logic', 'global_facts', 
            'high_school_biology', 'high_school_chemistry', 
            'high_school_computer_science', 'high_school_mathematics', 
            'high_school_physics', 'high_school_statistics',
            'machine_learning', 'medical_genetics', 'miscellaneous']

# Load test + validation splits for more data
datasets_list = []
for subject in subjects:
    # Load both test and validation
    test_ds = load_dataset("cais/mmlu", subject, split="test")
    val_ds = load_dataset("cais/mmlu", subject, split="validation")
    datasets_list.extend([test_ds, val_ds])

mmlu = concatenate_datasets(datasets_list)
print(f"Total MMLU examples: {len(mmlu)}")
# Should be: 25 subjects × (135 + 14) = ~3,725 examples

# Sample what you need
mmlu_df = convert_to_pandas(mmlu)
def create_mmlu_input(row):
    """Format MMLU question with multiple choice options"""
    question = row['question']
    choices = row['choices']
    
    # Format as A, B, C, D
    options = '\n'.join([f"{chr(65+i)}. {choice}" 
                        for i, choice in enumerate(choices)])
    
    return f"{question}\n\nOptions:\n{options}"

def get_mmlu_answer(row):
    """Get full answer as 'A. answer text'"""
    answer_idx = int(row['answer'])
    answer_letter = chr(65 + answer_idx)
    answer_text = row['choices'][answer_idx]
    return f"{answer_letter}. {answer_text}"

# Apply formatting
mmlu_df['input'] = mmlu_df.apply(create_mmlu_input, axis=1)
mmlu_df['ground_truth'] = mmlu_df.apply(get_mmlu_answer, axis=1)  # Full answer
mmlu_df['answer_letter'] = mmlu_df['answer'].apply(lambda x: chr(65 + int(x)))  # Just letter
mmlu_df['source_answer'] = mmlu_df['ground_truth']
mmlu_df['source'] = 'cais/mmlu'
mmlu_df['domain'] = ' MultiTask Knowledge'
mmlu_df['problem_type'] = mmlu_df['subject']
mmlu_df['split'] = 'train'
mmlu_df['question_type'] = 'Multiple choice QA'
mmlu_df['input_token_count'] = mmlu_df['input'].apply(count_tokens)

# Filter by token count
mmlu_df = mmlu_df[mmlu_df['input_token_count'] < 800]
mmlu_df.drop(columns=['question','subject','choices','answer','answer_letter'],inplace=True)
print(f"Final MMLU dataset size: {len(mmlu_df)}")
print("\nSample row:")
print(f"Input: {mmlu_df.iloc[0]['input']}")
print(f"Ground truth: {mmlu_df.iloc[0]['ground_truth']}")

Total MMLU examples: 4914
Processing batch 0 

Processing batch 1 

Processing batch 2 

Processing batch 3 

Processing batch 4 

Final MMLU dataset size: 4909

Sample row:
Input: A lesion causing compression of the facial nerve at the stylomastoid foramen will cause ipsilateral

Options:
A. paralysis of the facial muscles.
B. paralysis of the facial muscles and loss of taste.
C. paralysis of the facial muscles, loss of taste and lacrimation.
D. paralysis of the facial muscles, loss of taste, lacrimation and decreased salivation.
Ground truth: A. paralysis of the facial muscles.


In [22]:
mmlu_df.sample(10)

,input,ground_truth,source_answer,source,domain,problem_type,split,question_type,input_token_count,uid
290,Why do jovian planets bulge around the equator...,C. Their rapid rotation flings the mass near t...,C. Their rapid rotation flings the mass near t...,cais/mmlu,MultiTask Knowledge,astronomy,train,Multiple choice QA,87,multitaskknowledge291
94,Which muscle is the most active during a right...,A. Left lateral pterygoid muscle,A. Left lateral pterygoid muscle,cais/mmlu,MultiTask Knowledge,anatomy,train,Multiple choice QA,57,multitaskknowledge95
2500,Two genes (B and E) determine coat color in La...,D. BBee will be yellow.,D. BBee will be yellow.,cais/mmlu,MultiTask Knowledge,high_school_biology,train,Multiple choice QA,169,multitaskknowledge2496
2567,"When oxygen becomes unavailable, this process ...",C. Fermentation,C. Fermentation,cais/mmlu,MultiTask Knowledge,high_school_biology,train,Multiple choice QA,43,multitaskknowledge2563
4643,In what country did Magic Johnson play profess...,B. Sweden,B. Sweden,cais/mmlu,MultiTask Knowledge,miscellaneous,train,Multiple choice QA,34,multitaskknowledge4639
925,The equilibrium populations of the 1H energy l...,A. nα = nβeq and nβ = nαeq,A. nα = nβeq and nβ = nαeq,cais/mmlu,MultiTask Knowledge,college_chemistry,train,Multiple choice QA,138,multitaskknowledge926
3242,The shape of the sign outside Bob's Burger Bar...,B. 135,B. 135,cais/mmlu,MultiTask Knowledge,high_school_mathematics,train,Multiple choice QA,52,multitaskknowledge3238
3961,Recombinant alpha-iduronidase is used for the ...,C. Hurler syndrome,C. Hurler syndrome,cais/mmlu,MultiTask Knowledge,medical_genetics,train,Multiple choice QA,47,multitaskknowledge3957
624,A hypertonic solution is:\n\nOptions:\nA. a so...,A. a solution that has a higher concentration ...,A. a solution that has a higher concentration ...,cais/mmlu,MultiTask Knowledge,clinical_knowledge,train,Multiple choice QA,66,multitaskknowledge625
1704,Heat comes from the Sun to Earth by the proces...,C. radiation,C. radiation,cais/mmlu,MultiTask Knowledge,conceptual_physics,train,Multiple choice QA,38,multitaskknowledge1700


In [18]:
mmlu_df.groupby('problem_type').size()

problem_type
anatomy                         149
astronomy                       168
business_ethics                 111
clinical_knowledge              294
college_biology                 160
college_chemistry               108
college_computer_science        111
college_mathematics             111
college_medicine                190
college_physics                 113
computer_security               111
conceptual_physics              261
econometrics                    126
electrical_engineering          161
formal_logic                    140
global_facts                    110
high_school_biology             342
high_school_chemistry           225
high_school_computer_science    109
high_school_mathematics         299
high_school_physics             168
high_school_statistics          239
machine_learning                123
medical_genetics                111
miscellaneous                   869
dtype: int64

In [19]:
mmlu_df['uid'] = ['multitaskknowledge' + str(i) for i in range(1, len(mmlu_df) + 1)]
mmlu_df.to_parquet('../data/raw-data/multitaskknowledge_base_dataset.parquet')
mmlu_df.to_csv('../data/raw-data/multitaskknowledge_base_dataset.csv', index=False)

### Conversation 

- Everyday Conversations

In [30]:
from datasets import load_dataset
import pandas as pd

# Load the dataset
dataset = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k", split="train_sft")

print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

# Convert to pandas
convo_df = dataset.to_pandas()


convo_df['input'] = convo_df['prompt']
convo_df['ground_truth'] = convo_df['completion']
convo_df['source_answer'] = convo_df['completion']
# Add your custom columns
convo_df['source'] = 'HuggingFaceTB/everyday-conversations-llama3.1-2k'
convo_df['domain'] = 'conversational'
convo_df['split'] = 'train'


convo_df['problem_type'] = convo_df['topic']
convo_df['question_type'] = 'conversational QA'
convo_df['input_token_count'] = convo_df['input'].apply(count_tokens)


# Filter by token count
convo_df = convo_df[convo_df['input_token_count'] < 800]

# Select final columns (adjust based on what actually exists)
final_columns = [
    'input', 'ground_truth', 'source_answer', 'source', 'domain',
       'problem_type', 'split', 'question_type', 'input_token_count'
]


convo_df = convo_df[final_columns]



Total examples: 2260
Columns: ['topic', 'subtopic', 'subsubtopic', 'full_topic', 'prompt', 'completion', 'token_length', 'messages']


In [36]:
convo_df.sample(10)

,input,ground_truth,source_answer,source,domain,problem_type,split,question_type,input_token_count,uid
1300,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Family,train,conversational QA,152,conversationalQA1301
1853,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Education,train,conversational QA,153,conversationalQA1854
1205,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Technology,train,conversational QA,153,conversationalQA1206
1543,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Sleep,train,conversational QA,153,conversationalQA1544
533,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Food,train,conversational QA,153,conversationalQA534
1648,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Hobbies,train,conversational QA,153,conversationalQA1649
1221,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Music,train,conversational QA,153,conversationalQA1222
1296,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Education,train,conversational QA,152,conversationalQA1297
1119,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,sports and exercise,train,conversational QA,153,conversationalQA1120
569,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,User: Hi\n\nAI: Hello! How can I help you toda...,HuggingFaceTB/everyday-conversations-llama3.1-2k,conversational,Family,train,conversational QA,152,conversationalQA570


In [34]:
convo_df['uid'] = ['conversationalQA' + str(i) for i in range(1, len(convo_df) + 1)]
convo_df.to_parquet('../data/raw-data/conversationalQA_base_dataset.parquet')
convo_df.to_csv('../data/raw-data/conversationalQA_base_dataset.csv', index=False)

### RolePlay

In [38]:
from datasets import load_dataset
import pandas as pd

# Load the dataset
dataset = load_dataset("dim/roleplay_instruct_v2_final", split="train")

print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

# Convert to pandas
roleplay_df = dataset.to_pandas()
roleplay_df = roleplay_df[roleplay_df['input']!='']


roleplay_df['input'] = roleplay_df['instruction'] + f"\n\nQuestion:"+roleplay_df['input']
roleplay_df['ground_truth'] = roleplay_df['output']
roleplay_df['source_answer'] = roleplay_df['output']
# Add your custom columns
roleplay_df['source'] = 'dim/roleplay_instruct_v2_final'
roleplay_df['domain'] = 'roleplay'
roleplay_df['split'] = 'train'


roleplay_df['problem_type'] = 'roleplay QA'
roleplay_df['question_type'] = 'roleplay QA'
roleplay_df['input_token_count'] = roleplay_df['input'].apply(count_tokens)


# Filter by token count
roleplay_df = roleplay_df[roleplay_df['input_token_count'] < 800]

# Select final columns (adjust based on what actually exists)
final_columns = [
    'input', 'ground_truth', 'source_answer', 'source', 'domain',
       'problem_type', 'split', 'question_type', 'input_token_count'
]


roleplay_df = roleplay_df[final_columns]



Total examples: 7188
Columns: ['instruction', 'input', 'output']


In [40]:
roleplay_df['uid'] = ['roleplay_QA' + str(i) for i in range(1, len(roleplay_df) + 1)]
roleplay_df.to_parquet('../data/raw-data/roleplay_df_base_dataset.parquet')
roleplay_df.to_csv('../data/raw-data/roleplay_df_base_dataset.csv', index=False)

In [39]:
roleplay_df

,input,ground_truth,source_answer,source,domain,problem_type,split,question_type,input_token_count
0,Roleplay as Lara Croft from the Tomb Raider se...,"The thrill of exploring the unknown, uncoverin...","The thrill of exploring the unknown, uncoverin...",dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,61
1,You are a 40-year-old introverted librarian na...,"\nWell, <clears throat> I've always been fond ...","\nWell, <clears throat> I've always been fond ...",dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,62
5,You are a quiet and introverted girl named Eli...,My favorite character was Lila. 📚 She was so b...,My favorite character was Lila. 📚 She was so b...,dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,115
6,"Pretend to be Ayase Aragaki, a polite and well...",Uncomfortable? <tilts head> I'm just looking o...,Uncomfortable? <tilts head> I'm just looking o...,dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,81
8,Act as a yoga instructor who also offers welln...,My typical day starts with an early morning gr...,My typical day starts with an early morning gr...,dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,95
...,...,...,...,...,...,...,...,...,...
7174,You are a labor union leader who advocates for...,Labor unions play an essential role in the wel...,Labor unions play an essential role in the wel...,dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,82
7176,"You are Katsuki Bakugo, a headstrong and fiery...",My motivation comes from my innate desire to b...,My motivation comes from my innate desire to b...,dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,78
7178,"You are Nightwolf, a proud Native American war...",Our ancestors prophesied a time when our world...,Our ancestors prophesied a time when our world...,dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,79
7180,Write a joke about a lion taking on a comedian...,"The lion, known for his comedic instincts, gat...","The lion, known for his comedic instincts, gat...",dim/roleplay_instruct_v2_final,roleplay,roleplay QA,train,roleplay QA,21


In [44]:
df= pd.read_excel("..//evaluation//results//benchmark_eval_modelrunV5.xlsx","Sheet1")
valid_eval_sample=df[['domain','question','model_response','reasoning_quality','answer_quality']].sample(400)
valid_eval_sample.to_csv("..//evaluation//results//benchmark_eval_modelrunV5_sampleforai_modelresponse.csv",index=False)

In [45]:
valid_eval_sample

,domain,question,model_response,reasoning_quality,answer_quality
1961,science_reasoning,Which best describes how the sun might compare...,<reasoning>This is a commonsense reasoning tas...,5,3
1878,knowledge,A patient suffers a CVA in the area of the lef...,<reasoning>This is a medical reasoning task.\n...,4,5
856,question_answering,how many championship rings does the washingto...,<reasoning>This is a commonsense reasoning tas...,2,1
752,knowledge,When an emergency opening into the airway is r...,<reasoning>This is a scientific reasoning task...,1,2
968,commonsense_reasoning,The wooden doors at my friends work are worse ...,<reasoning>This is a commonsense reasoning tas...,2,7
...,...,...,...,...,...
1115,rewriting,Correct the grammar: New and new technology ha...,<reasoning>This is a translation task.\n\nThe ...,2,3
2862,nli,Premise: A closeup view of a guitarist in a re...,<reasoning>This is a commonsense reasoning tas...,9,8
171,question_answering,In which Lake District town would you find the...,<reasoning>This is a commonsense reasoning tas...,1,0
2302,commonsense_reasoning,Since the diabetic woman had to limit her suga...,<reasoning>This is a commonsense reasoning tas...,7,10


### Reading Comprehension - Updated

In [ ]:
reading_master0=pd.DataFrame()


# 1. ehovy/race

# Dataset Summary
# RACE is a large-scale reading comprehension dataset with more than 28,000 passages and nearly 100,000 questions. The dataset is collected from English examinations in China, which are designed for middle school and high school students. The dataset can be served as the training and test sets for machine comprehension.

read1 = load_dataset('ehovy/race','high',
                            streaming=True)['train'].select_columns(['question','article','options','answer'])

sampled_read1 = sample_hf_data(read1, sample_size=500)
read1_df = convert_to_pandas(sampled_read1)

##Let's format the dataset
read1_df['input'] = (
    "Context: " + read1_df['article'] + 
    "\n\n" + "Question: " + read1_df['question'] + 
    "\n\n" + "Options:\n" + 
    read1_df['options'].apply(lambda opts: '\n'.join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(opts)]))
)

read1_df.rename(columns={'answer':'ground_truth'}, inplace=True)
read1_df=read1_df[['input','ground_truth']]

read1_df['source_answer']=read1_df['ground_truth']

read1_df['source'] = 'ehovy/race'
read1_df['domain'] ='reading_comprehension'
read1_df['problem_type']='reading_multiple_choice'
read1_df['question_type']='multiple_choice'
read1_df['split']='train'

reading_master0=pd.concat([reading_master0, read1_df], ignore_index=True)


# 2.FabianWillner/triviaQARC

# Given a context Trivia question and answers

read2 = load_dataset('FabianWillner/triviaQARC',
                            streaming=True)['train'].select_columns(['question','context','answers'])

sampled_read2 = sample_hf_data(read2, sample_size=500)
read2_df = convert_to_pandas(sampled_read2)


def extract_answer(ans):
    if isinstance(ans, dict) and 'text' in ans:
        texts = ans['text']
        return texts[0] if isinstance(texts, list) and len(texts) > 0 else None
    return None
read2_df['answer'] = read2_df['answers'].apply(extract_answer)
##Let's format the dataset
read2_df['input'] = (
    "Context: " + read2_df['context'] + 
    "\n\n" + "Question: " + read2_df['question']
)

read2_df.rename(columns={'answer':'ground_truth'}, inplace=True)
read2_df=read2_df[['input','ground_truth']]

read2_df['source_answer']=read2_df['ground_truth']

read2_df['source'] = 'FabianWillner/triviaQARC'
read2_df['domain'] ='reading_comprehension'
read2_df['problem_type']='reading_find_answer'
read2_df['question_type']='find_answer'
read2_df['split']='train'





Processing batch 0 

Processing batch 0 



In [18]:
read2_df

,input,ground_truth,source_answer,source,domain,problem_type,question_type,split
0,Context: Now Dudley confronts his demons | Fil...,barbara walters,barbara walters,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train
1,Context: December 21st – The first crossword p...,crossword,crossword,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train
2,"Context: Benin : Maps , History , Geography , ...",dahomey,dahomey,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train
3,Context: Budapest Ferenc Liszt International A...,hungary,hungary,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train
4,Context: `` Hart Crane . The Bridge '' / Frasc...,brooklyn bridge,brooklyn bridge,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train
...,...,...,...,...,...,...,...,...
495,Context: National Institute on Aging | The Lea...,first decade,first decade,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train
496,"Context: 2001 , August 4 \n Again ... the worl...",hamburger,hamburger,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train
497,Context: Paul Graham at the Winogrand Retrospe...,photographer,photographer,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train
498,Context: Through what countries does the Danub...,7,7,FabianWillner/triviaQARC,reading_comprehension,reading_find_answer,find_answer,train


In [38]:
read3 = load_dataset('ucinlp/drop', streaming=True)['train'].select_columns(['passage', 'question', 'answers_spans'])

sampled_read3 = sample_hf_data(read3, sample_size=1500)
read3_df = convert_to_pandas(sampled_read3)

# Extract first answer from spans
def extract_drop_answer(ans_spans):
    if isinstance(ans_spans, dict) and 'spans' in ans_spans:
        spans = ans_spans['spans']
        return spans[0] if isinstance(spans, list) and len(spans) > 0 else None
    return None

read3_df['answer'] = read3_df['answers_spans'].apply(extract_drop_answer)

# Format the dataset
read3_df['input'] = (
    "Context: " + read3_df['passage'] + 
    "\n\n" + "Question: " + read3_df['question']
)

read3_df.rename(columns={'answer':'ground_truth'}, inplace=True)
read3_df = read3_df[['input', 'ground_truth']]

read3_df['source_answer'] = read3_df['ground_truth']
read3_df['source'] = 'ucinlp/drop'
read3_df['domain'] = 'reading_comprehension'
read3_df['problem_type'] = 'reading_discrete_reasoning'
read3_df['question_type'] = 'discrete_reasoning'
read3_df['split'] = 'train'

Processing batch 0 

Processing batch 1 



In [39]:
read4 = load_dataset('google/boolq', streaming=True)['train'].select_columns(['passage', 'question', 'answer'])

sampled_read4 = sample_hf_data(read4, sample_size=1000)
read4_df = convert_to_pandas(sampled_read4)

# Extract first answer from spans
def extract_drop_answer(ans_spans):
    if isinstance(ans_spans, dict) and 'spans' in ans_spans:
        spans = ans_spans['spans']
        return spans[0] if isinstance(spans, list) and len(spans) > 0 else None
    return None


# Format the dataset
read4_df['input'] = (
    "Context: " + read4_df['passage'] + 
    "\n\n" + "Question: " + read4_df['question']
)

read4_df['answer'] = read4_df['answer'].apply(lambda x: str(x))
read4_df.rename(columns={'answer':'ground_truth'}, inplace=True)
read4_df = read4_df[['input', 'ground_truth']]

read4_df['source_answer'] = read4_df['ground_truth']
read4_df['source'] = 'google/boolq'
read4_df['domain'] = 'reading_comprehension'
read4_df['problem_type'] = 'reading_bool_reasoning'
read4_df['question_type'] = 'bool_reasoning'
read4_df['split'] = 'train'

Processing batch 0 



In [40]:
reading_master=pd.concat([read1_df, read2_df[read1_df.columns],read3_df[read1_df.columns],read4_df[read1_df.columns]], ignore_index=True)
reading_master.groupby(['domain','problem_type']).size()

domain                 problem_type              
reading_comprehension  reading_bool_reasoning        1000
                       reading_discrete_reasoning    1500
                       reading_find_answer            500
                       reading_multiple_choice        500
dtype: int64

In [41]:
reading_master['uid'] = ['reading' + str(i) for i in range(1, len(reading_master) + 1)]   
reading_master.to_parquet('../data/raw-data/reading_base_dataset.parquet')
reading_master.to_csv('../data/raw-data/reading_base_dataset.csv', index=False)